In [57]:
import random
import os
import hashlib
from pathlib import Path
import time

file_size = 13
signature_size = 10
container_size = 31

file_path = "vo_13bit.bin"
container_path = "mem_chunk.bin"
signature_path = "signature.bin"



In [58]:
def create_random_file():
    num = random.getrandbits(file_size)

    with open(file_path, "wb") as f:
        f.write(num.to_bytes(2, "big"))

    print(f"Создан {file_path} — {file_size} случайных бит (на диске до целых байт).")
    print("Значение (int):", num, "битовое:", bin(num)[2:].zfill(file_size))


def create_container():
    # size = container_size * 1024 * 1024
    size = container_size * 1024 * 1024
    with open(container_path, "wb") as f:
        f.write(os.urandom(size))
    print(f"Создан {container_path} — {container_size} случайных Мбайт (на диске до целых Мбайт).")


def create_signature_from_file():
    file = Path(file_path)
    if not file.exists():
        raise FileNotFoundError(f"Файл не найден: {file_path}")

    data = file.read_bytes()
    h = hashlib.sha256(data).digest()
    sig = h[:signature_size]

    with open(signature_path, "wb") as f:
        f.write(sig)

    return sig



In [ ]:
def count_coincidence(precent):
    sig_bytes = Path(signature_path).read_bytes()
    cont_bytes = Path(container_path).read_bytes()

    sig_len_bits = len(sig_bytes) * 8
    cont_len_bits = len(cont_bytes) * 8
    hits = 0

    total_shifts = cont_len_bits - sig_len_bits + 1
    if total_shifts <= 0:
        return 0


    sig_int = int.from_bytes(sig_bytes, "big")

    report_interval = max(1, total_shifts // 100)
    report_interval = max(report_interval, 1000)

    start_time = time.time()

    for shift in range(total_shifts):
    
        start_byte = shift // 8
        end_byte = (shift + sig_len_bits + 7) // 8 
        window_bytes = cont_bytes[start_byte:end_byte]
        window_int = int.from_bytes(window_bytes, "big")

    
        shift_amount = (8 * len(window_bytes) - sig_len_bits - (shift % 8))
        window_int = (window_int >> shift_amount) & ((1 << sig_len_bits) - 1)

    
        mismatch_bits = (sig_int ^ window_int).bit_count()
        match_count = sig_len_bits - mismatch_bits

        if match_count * 100 / sig_len_bits >= precent:
            hits += 1

    
        if shift % report_interval == 0 or shift == total_shifts - 1:
            now = time.time()
            elapsed = now - start_time
            done_pct = (shift + 1) * 100 / total_shifts
            if shift > 0:
                est_total = elapsed * total_shifts / (shift + 1)
                remaining = max(0, est_total - elapsed)
                rem_s = int(remaining)
            else:
                rem_s = -1
            print(f"\rProgress: {int(done_pct)}% ({shift+1}/{total_shifts})  elapsed: {int(elapsed)}s  rem ~ {rem_s}s", end="", flush=True)

    print()
    total_time = time.time() - start_time
    print(f"Done. hits={hits}. Time: {int(total_time)}s")
    return hits


In [ ]:
create_container()
create_random_file()
create_signature_from_file()

for i in range(100, 50, -1):
    print(f"For {i}%")
    c = count_coincidence(i)
    if c != 0:
        print(f"Первое ложное срабатываение возникло при {i}%, количество: {c}")
        break


Создан mem_chunk.bin — 31 случайных Мбайт (на диске до целых Мбайт).
Создан vo_13bit.bin — 13 случайных бит (на диске до целых байт).
Значение (int): 7170 битовое: 1110000000010
For 100% 
For 100%  0% (1/260046769)  elapsed: 0s  rem ~ -1s
For 100%  1% (2600468/260046769)  elapsed: 0s  rem ~ 90s
For 100%  1% (5200935/260046769)  elapsed: 1s  rem ~ 90s
For 100%  2% (7801402/260046769)  elapsed: 2s  rem ~ 89s
For 100%  3% (10401869/260046769)  elapsed: 3s  rem ~ 88s
For 100%  4% (13002336/260046769)  elapsed: 4s  rem ~ 87s
For 100%  5% (15602803/260046769)  elapsed: 5s  rem ~ 86s
Progress: 6% (18203270/260046769)  elapsed: 6s  rem ~ 86s

KeyboardInterrupt: 